In [2]:
import polars as pl
import altair as alt 
import geopandas as gpd

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [3]:
# Import data 

from_2016_to_now_df = pl.read_parquet("data/cpd_activities_2015-12-31_to_now_detailed.parquet")

In [12]:
def graph_yearly_activities(df, title, percent=True):
    AGE_GROUPS = {
        # early childhood: at least 3 yrs but less than 5 yrs
        # youth: at least 8 yrs but less than 13 yrs
        # teen: at least 13 yrs but less than 20 yrs
        # young adult: at least 18 but less than 29
        # adult: 18 yrs +
        # senior: 60 +
        "Early Childhood": "Kids & Teens",
        "Youth":           "Kids & Teens",
        "Teen":            "Kids & Teens",
        "Young Adult":     "Adults",
        "Adult":           "Adults",
        "Senior":          "Seniors",
        "All Ages":        "All Ages",
    }

    yearly_grouped = (
        df.group_by(["season_year", "category"])
          .len()
          .with_columns(age_group=pl.col("category").replace_strict(AGE_GROUPS))
          .group_by(["season_year", "age_group"])
          .agg(pl.col("len").sum())
          .with_columns(
              pct=pl.col("len") / pl.col("len").sum().over("season_year") * 100
          )
          .sort(["season_year", "age_group"])
          .filter(~pl.col("season_year").is_in([2015, 2020, 2021, 2026]))
    )

    if percent:
        y = alt.Y('pct:Q', title='% of CPD activities',
                  scale=alt.Scale(domain=[0, 100]))
    else:
        y = alt.Y('len:Q', title='No. of CPD activities')

    return alt.Chart(yearly_grouped).mark_line(point=True).encode(
        x=alt.X('season_year:O', title='Year', axis=alt.Axis(labelAngle=0)),
        y=y,
        color=alt.Color('age_group:N', title="Age groups"),
        tooltip=[
            'season_year:O',
            'age_group:N',
            'len:Q',
            alt.Tooltip('pct:Q', format='.1f'),
        ],
    ).properties(width=600, height=350, title=title)

In [13]:
graph_yearly_activities(from_2016_to_now_df, 
                        "Share of Chicago Park District activities by age group")

alt.Chart(...)

In [ ]:
# Filtering to activities with at least four sessions to weed out events

from_2016_to_now_longer = from_2016_to_now_df.with_columns(
    date_start_parsed=pl.col("date_start").str.to_date(strict=False),
    date_end_parsed=pl.col("date_end").str.to_date(strict=False),
    ).with_columns(
        date_end_parsed=pl.coalesce("date_end_parsed", "date_start_parsed"),
    ).with_columns(
        duration_days=(pl.col("date_end_parsed") - pl.col("date_start_parsed"))
                        .dt.total_days() + 1,                     
        meetings_per_week=pl.when(pl.col("days_of_week").is_null()
                                | (pl.col("days_of_week") == ""))
                            .then(None)
                            .otherwise(pl.col("days_of_week").str.split(",").list.len()),
    ).with_columns(
        n_sessions_est=(pl.col("duration_days") / 7 * pl.col("meetings_per_week"))
                        .round(0).cast(pl.Int32),
    ).filter(pl.col("n_sessions_est") >= 4)


graph_yearly_activities(from_2016_to_now_longer, 
                        "Share of Chicago Park District activities by age group (excluding events)")

alt.Chart(...)

##### Find source - cancelled senior sessions

In [8]:
cancelled_parks = (
    from_2016_to_now_longer
    .filter(
        (
            (pl.col("category") == "Senior") & 
            (pl.col("status") == "Cancelled") & 
            (~pl.col("season_year").is_in([2020,2021]))
        )
    )
    .group_by("park")
    .agg(
       n_cancelled=pl.len()
    )    
).sort(by="n_cancelled", descending=True)

cancelled_parks.head(10)

park,n_cancelled
str,u32
"""Don Nash Cmty Ctr""",18
"""Ellis Park""",15
"""Hayes Park""",14
"""Lincoln Park Cultural Ctr""",13
"""Maggie Daley Park""",13
"""Independence Park""",10
"""Mann Park""",10
"""Dvorak Park""",8
"""Foster Park""",7


In [10]:
from_2016_to_now_longer.filter(
    (
        (pl.col("category") == "Senior") & 
        (pl.col("status") == "Cancelled") &
        (~pl.col("season_year").is_in([2020,2021])) &
        (pl.col("park") == "Don Nash Cmty Ctr")
     )
)

activity_name,park,ages,age_description,date_start,date_end,date_range,time,days_of_week,category,fee,status,activity_id,detail_url,center_address,center_zip,latitude,longitude,date_start_parsed,season_inferred,season_year,date_end_parsed,duration_days,meetings_per_week,n_sessions_est
str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,f64,f64,date,str,i32,date,i64,u32,i32
"""Aquatic Exercise II - Low Impa…","""Don Nash Cmty Ctr""","""60 and up""","""60 yrs +,""","""2022-09-16""","""2022-10-14""","""September 16, 2022 to October …","""10:00 AM - 11:00 AM""","""Fri""","""Senior""","""Free""","""Cancelled""",436243,"""https://apm.activecommunities.…","""1833 E. 71st St.""","""60649""",41.7659,-87.58,2022-09-16,"""Fall""",2022,2022-10-14,29,1,4
"""Aquatic Exercise II - Low Impa…","""Don Nash Cmty Ctr""","""60 and up""","""60 yrs +,""","""2022-09-16""","""2022-10-14""","""September 16, 2022 to October …","""9:00 AM - 10:00 AM""","""Fri""","""Senior""","""Free""","""Cancelled""",436244,"""https://apm.activecommunities.…","""1833 E. 71st St.""","""60649""",41.7659,-87.58,2022-09-16,"""Fall""",2022,2022-10-14,29,1,4
"""Aquatic Exercise II - Low Impa…","""Don Nash Cmty Ctr""","""60 and up""","""60 yrs +,""","""2022-09-12""","""2022-10-10""","""September 12, 2022 to October …","""9:00 AM - 10:00 AM""","""Mon""","""Senior""","""Free""","""Cancelled""",436246,"""https://apm.activecommunities.…","""1833 E. 71st St.""","""60649""",41.7659,-87.58,2022-09-12,"""Fall""",2022,2022-10-10,29,1,4
"""Aquatic Exercise II - Low Impa…","""Don Nash Cmty Ctr""","""60 and up""","""60 yrs +,""","""2022-09-15""","""2022-10-13""","""September 15, 2022 to October …","""9:00 AM - 10:00 AM""","""Thu""","""Senior""","""Free""","""Cancelled""",436247,"""https://apm.activecommunities.…","""1833 E. 71st St.""","""60649""",41.7659,-87.58,2022-09-15,"""Fall""",2022,2022-10-13,29,1,4
"""Aquatic Exercise II - Low Impa…","""Don Nash Cmty Ctr""","""60 and up""","""60 yrs +,""","""2022-09-14""","""2022-10-12""","""September 14, 2022 to October …","""9:00 AM - 10:00 AM""","""Wed""","""Senior""","""Free""","""Cancelled""",436248,"""https://apm.activecommunities.…","""1833 E. 71st St.""","""60649""",41.7659,-87.58,2022-09-14,"""Fall""",2022,2022-10-12,29,1,4
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Aquatic Exercise III - High Im…","""Don Nash Cmty Ctr""","""60 and up""","""60 yrs +,""","""2022-09-15""","""2022-10-13""","""September 15, 2022 to October …","""11:00 AM - Noon""","""Thu""","""Senior""","""Free""","""Cancelled""",421819,"""https://apm.activecommunities.…","""1833 E. 71st St.""","""60649""",41.7659,-87.58,2022-09-15,"""Fall""",2022,2022-10-13,29,1,4
"""Chair Yoga at Don Nash""","""Don Nash Cmty Ctr""","""60 and up""","""60 yrs +,""","""2023-09-08""","""2023-12-08""","""September 8, 2023 to December …","""11:00 AM - Noon""","""Fri""","""Senior""","""Free""","""Cancelled""",465838,"""https://apm.activecommunities.…","""1833 E. 71st St.""","""60649""",41.7659,-87.58,2023-09-08,"""Fall""",2023,2023-12-08,92,1,13
"""Chair Yoga at Don Nash""","""Don Nash Cmty Ctr""","""60 and up""","""60 yrs +,""","""2024-10-09""","""2024-12-11""","""October 9, 2024 to December 11…","""9:30 AM - 10:30 AM""","""Wed""","""Senior""","""Free""","""Cancelled""",502634,"""https://apm.activecommunities.…","""1833 E. 71st St.""","""60649""",41.7659,-87.58,2024-10-09,"""Fall""",2024,2024-12-11,64,1,9


In [14]:
graph_yearly_activities(
    from_2016_to_now_longer.filter(pl.col("status") == "Cancelled"),
    "No. of cancelled CPD activities by age group",
    percent=False,
)

alt.Chart(...)

In [16]:
graph_yearly_activities(
    from_2016_to_now_longer.filter(
        (pl.col("status") == "Cancelled") & 
        (pl.col("category") == "Senior")
    ),
    "No. of cancelled CPD activities for seniors",
    percent=False,
)

alt.Chart(...)